In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \mathbf{z} = (z_0, \dots, z_c), \quad c \in \mathcal{C}, \quad z_c \in \mathbb{R} $$

$$ p_c = \frac{e^{z_c}}{\sum_{k} e^{z_k}} $$

$$
\frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j) =
\begin{dcases}
i = j & \frac{\partial p_i}{\partial z_i} = 
\frac{e^{z_i} \sum_{k} e^{z_k} - e^{z_i} e^{z_i}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} =
p_i (1 - p_i) \\
\\
i \neq j & \frac{\partial p_i}{\partial z_j} = 
\frac{0 \cdot \sum_{k} e^{z_k} - e^{z_i} e^{z_j}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} = 
-p_i p_j  = p_i (0 - p_j)\\
\end{dcases}
$$

$$
J = \frac{\partial \mathbf{p}}{\partial \mathbf{z}} = \begin{pmatrix}
p_0 (1 - p_0) & -p_0 p_1 & \cdots & -p_0 p_c \\
-p_1 p_0 & p_1 (1 - p_1) & \cdots & -p_1 p_c \\
\vdots & \vdots & \ddots & \vdots \\
-p_c p_0 & -p_c p_1 & \cdots & p_c (1 - p_c) \\
\end{pmatrix}
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _softmax(x: tr.Tensor) -> tr.Tensor:
    ex = tr.exp(x)
    y = ex / tr.sum(ex)
    return y


def softmax(x: tr.Tensor) -> tr.Tensor:
    x= tr.as_tensor(x)
    return _softmax(x - tr.max(x)) 


class SoftmaxFunction(autograd.Function):
    """Custom autograd function for the `softmax`."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        p = softmax(z)
        ctx.save_for_backward(z)
        return p
    
    @staticmethod
    def backward(ctx, grad_output: tr.Tensor) -> tuple[tr.Tensor,]:
        (z, ) = ctx.saved_tensors
        S = z.shape[0]

        p = softmax(z)
        assert p.shape == (S, )

        assert grad_output.shape == (S, )

        diag_p = tr.diag(p)
        assert diag_p.shape == (S, S)

        ppt = p.unsqueeze(1) * p.unsqueeze(0)
        assert ppt.shape == (S, S)

        J = diag_p - ppt
        assert J == approx(J.t())

        grad_input = J @ grad_output

        # # Jacobian-vector product: (diag(p) - p p^T) @ grad_output
        # dot = (grad_output * p).sum()
        # grad_input = p * (grad_output - dot)

        return (grad_input, )
    

class Softmax(nn.Module):
    """Custom module for the `softmax`."""
    
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(x)

In [ ]:
# Unit tests

def test_basic() -> None:
    assert softmax(1000) == approx(1.0)
    assert softmax(100) == approx(1.0)
    assert softmax(10) == approx(1.0)
    assert softmax(1) == approx(1.0)


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


# def test_compare(samples) -> None:
#     x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

#     x1 = x.clone().detach().requires_grad_(True)
#     y1 = Softmax()(x1)
#     F1 = y1.sum()
#     F1.backward()

#     x2 = x.clone().detach().requires_grad_(True)
#     y2 = tr.sigmoid(x2)
#     F2 = y2.sum()
#     F2.backward()

#     assert y1 == approx(y2)
#     assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    # test_compare(1)
    # test_compare(10)
    # test_compare(100)